# v10 Kaggle 提出ノート(推論のみ)

学習はローカル(train_models_v10.ipynb)。ここでは **test 特徴量を GPU PF(v10パイプライン)で構築** →
学習済みモデル(LightGBM/CatBoost/XGBoost ×3)をロード → アンサンブル → 補正(apply_pp+sg_smooth)→ submission.csv。

必要な Kaggle データセット(input):
- コンペデータ(train/ test/ sample_submission.csv あり)
- v10コード: `create_train_parquet_v10.py`(これを import して `build_v10_dataset` を使う)
- v9特徴量ノート: `pf_tuning_create_train_csv_multi_pf_v9.ipynb`(build_well を exec で再利用)
- 学習済みモデル: `artifacts_v10_2/models/{lightgbm,catboost,xgboost}-1..3/*.pkl`
- koolbox(wheel か ソース)
★ test特徴は train と同じPF設定(N_PF_SETS/FWD_SEEDS/BWD_PER_SEED)で作ること(列を一致させる)。

In [ ]:
# koolbox セットアップ(wheel install か sys.path)
import sys, os, glob, subprocess
kb_dir = "/kaggle/input/koolbox-offline"
if not os.path.isdir(kb_dir):
    cand = glob.glob("/kaggle/input/**/koolbox*", recursive=True)
    if cand: kb_dir = cand[0]
print("koolbox dir:", kb_dir)
if os.path.isdir(kb_dir):
    whls = glob.glob(f"{kb_dir}/**/*.whl", recursive=True)
    if whls:
        for w in whls: subprocess.run(["pip","install","--no-deps",w], check=False)
    else:
        sys.path.insert(0, kb_dir)
        for sub in os.listdir(kb_dir):
            p=os.path.join(kb_dir,sub)
            if os.path.isdir(p): sys.path.insert(0,p)
import koolbox; print("koolbox OK:", koolbox.__file__)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, joblib, gc, os, pathlib, torch
from scipy.signal import savgol_filter
from sklearn.metrics import root_mean_squared_error
if os.name != "nt": pathlib.WindowsPath = pathlib.PosixPath
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# ===== パス設定(Kaggle) — 実際のデータセット名に合わせて調整 =====
class CFG:
    dataset_path   = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")   # train/ test/ あり
    v10_code_dir   = Path("/kaggle/input/rogii-v10-code")        # create_train_parquet_v10.py がある場所
    v9_nb_path     = Path("/kaggle/input/rogii-v9-nb/pf_tuning_create_train_csv_multi_pf_v9.ipynb")
    models_path    = Path("/kaggle/input/rogii-artifact-v10/artifacts_v10_2")   # models/ がある
    metric = root_mean_squared_error
# 学習時と同じPF設定(列一致のため必須)
N_PF_SETS, FWD_SEEDS, BWD_PER_SEED, WELL_CHUNK, N_JOBS_CPU = 3, 16, 8, 130, 4
BUILD_BACKWARD = True   # 順方向のみ提出なら False(+ FWD_SEEDS=128, models_path を順方向版に)
pp_params = {"alpha":1.0, "tau":85, "w_pf":0.09}

In [ ]:
# ===== v10 パイプラインを import し、Kaggleパスに上書きして test 特徴を構築 =====
sys.path.insert(0, str(CFG.v10_code_dir))
import create_train_parquet_v10 as v10m

v10m.DATA_DIR    = CFG.dataset_path           # train/(FI/DI fit用)と test/ を含む
v10m.V9_NB       = CFG.v9_nb_path
v10m.DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
v10m.N_PF_SETS   = N_PF_SETS
v10m.FWD_SEEDS   = FWD_SEEDS
v10m.BWD_PER_SEED= BWD_PER_SEED
v10m.WELL_CHUNK  = WELL_CHUNK
v10m.BUILD_BACKWARD = BUILD_BACKWARD
v10m.N_JOBS_CPU  = N_JOBS_CPU

V9 = v10m.load_v9_namespace()                 # FI/DI を train で fit + build_well 取得
test_df = v10m.build_v10_dataset(V9, split="test", is_train=False)
print("test_df:", test_df.shape)
DROP = {"well","id","target"}

In [ ]:
# cmp_(比較特徴)は学習で使っていないので test からも除去 → 列数を学習(627)と一致させる
cmp_cols = [c for c in test_df.columns if c.startswith("cmp_")]
test_df = test_df.drop(columns=cmp_cols)
print("dropped", len(cmp_cols), "cmp_ cols ->", test_df.shape)

In [ ]:
# ===== モデルのロード補助(v9提出ノートと同じ:学習時の特徴量順を取得)=====
def _as_list(v):
    if v is None: return None
    if callable(v):
        try: v=v()
        except TypeError: return None
    if isinstance(v,(list,tuple,np.ndarray,pd.Index)):
        out=[str(x) for x in list(v)]; return out or None
    return None
def _iter_models(o):
    yield o
    for a in ["models","models_","estimators_","fold_models","fold_models_","model","estimator","estimator_","best_estimator_"]:
        if hasattr(o,a):
            v=getattr(o,a)
            if isinstance(v,dict): yield from v.values()
            elif isinstance(v,(list,tuple)): yield from v
            elif v is not None: yield v
def feat_cols_from_trainer(tr):
    for a in ["features","feature_cols","feature_columns","feature_names","feature_names_","columns","cols","input_features"]:
        if hasattr(tr,a):
            c=_as_list(getattr(tr,a))
            if c: return c
    for m in _iter_models(tr):
        for a in ["feature_names_in_","feature_name_","feature_names_","feature_names"]:
            if hasattr(m,a):
                c=_as_list(getattr(m,a))
                if c: return c
        if hasattr(m,"get_booster"):
            try:
                c=_as_list(m.get_booster().feature_names)
                if c: return c
            except Exception: pass
        if hasattr(m,"feature_names_"):
            c=_as_list(getattr(m,"feature_names_"))
            if c: return c
    return [c for c in test_df.columns if c not in DROP]
def make_X(tr):
    f=feat_cols_from_trainer(tr); miss=[c for c in f if c not in test_df.columns]
    if miss: raise ValueError(f"test_df missing {len(miss)} feats: {miss[:15]}")
    return test_df[f]

In [ ]:
# ===== 9モデルをロードして test 予測 → アンサンブル(平均)=====
test_preds=[]
for fam in ["lightgbm","catboost","xgboost"]:
    for i in range(1,4):
        d = CFG.models_path / "models" / f"{fam}-{i}"
        pkls = sorted(d.glob("*.pkl"))
        if not pkls: raise FileNotFoundError(f"no pkl in {d}")
        tr = joblib.load(pkls[0])
        pred = tr.predict(make_X(tr))
        test_preds.append(np.asarray(pred, dtype=np.float32))
        print(f"loaded {fam}-{i}")
        del tr; gc.collect()
print("n models:", len(test_preds))

In [ ]:
# ===== 補正(v9と同じ apply_pp + sg_smooth)=====
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md*(1-w_pf) + pd_*w_pf
    if tau:
        d *= (1.-np.exp(-np.maximum(df["md_since"].values,0.)/tau))
    return d*alpha
def sg_smooth(df, col, sg_w=17, sg_p=3):
    df=df.copy()
    for _,gg in df.groupby("well", sort=False):
        v=gg[col].values; n=len(v); wl=min(sg_w,n)
        if wl%2==0: wl-=1
        if wl>=sg_p+2: v=savgol_filter(v,wl,sg_p)
        df.loc[gg.index,col]=v
    return df

In [ ]:
# ===== 推論(アンサンブル → 補正)=====
test_df2 = test_df.copy()
pf_test = test_df2["pf_ancc"].values - test_df2["last_known_tvt"].values
test_df2["pred"] = test_df2["last_known_tvt"].values + apply_pp(
    test_df2, np.mean(test_preds, axis=0), pf_test, **pp_params)
test_df2 = sg_smooth(test_df2, "pred")

In [ ]:
# ===== 提出 =====
sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
sub = sample_sub[["id"]].merge(test_df2[["id","pred"]].rename(columns={"pred":"tvt"}), on="id", how="left")
sub["tvt"] = sub["tvt"].fillna(float(np.nanmedian(test_df2["last_known_tvt"].values)))
sub[["id","tvt"]].to_csv("submission.csv", index=False)
print("saved submission.csv", sub.shape); sub.head()